# Reproduce Paper Results (No Re-Simulation)

This notebook regenerates **all numeric tables and experimental figures** from the EnsembleNet paper using frozen result files in `results/`.

It does **not** download Kaggle data, train models, or re-run ride pooling.

**Paper:** `Taxi_Clubbing (1)/main.tex`

**One-shot alternative:**
```bash
python scripts/regenerate_all.py
```

**Single artefact:**
```bash
python scripts/regenerate_all.py --only fig:model_comp_full
python scripts/regenerate_all.py --only tab:results_full
```


## Setup

Imports the regenerator and confirms frozen data is present.


In [ ]:
from pathlib import Path
import sys

ROOT = Path('.').resolve()
if not (ROOT / 'results' / 'model_results.csv').exists():
    # allow running from scripts/ or repo root
    ROOT = Path('..').resolve() if (Path('..') / 'results' / 'model_results.csv').exists() else ROOT

sys.path.insert(0, str(ROOT / 'scripts'))
import regenerate_all as regen

regen.ROOT = ROOT
regen.RESULTS = ROOT / 'results'
regen.FIG_DIR = ROOT / 'reproduced_figures'
regen.TAB_DIR = ROOT / 'reproduced_tables'
regen._ensure_dirs()

print('Repo root :', ROOT)
print('Results   :', regen.RESULTS)
print('Figures → :', regen.FIG_DIR)
print('Tables  → :', regen.TAB_DIR)
print('Frozen files:', len(list(regen.RESULTS.glob('*'))))


---
## Demand prediction

### Cell → `tab:results_full` (Table: 14-model test metrics)

Source: `results/model_results.csv`


In [ ]:
df = regen.table_results_full()
df

### Cell → `fig:model_comp_full`

4-panel RMSE / R² / MAPE / ensemble-vs-individual box plot.

Source: `results/model_results.csv`  
Outputs: `reproduced_figures/fig_model_comparison_full.{pdf,png}`


In [ ]:
regen.fig_model_comparison_full()
from IPython.display import Image, display
display(Image(filename=str(regen.FIG_DIR / 'fig_model_comparison_full.png')))

### Cell → `tab:kfold`

Source: `results/kfold_cv_summary.csv`


In [ ]:
regen.table_kfold()

---
## Statistical significance

### Cell → `fig:significance` (+ Wilcoxon CSV)

Sources:
- `results/cv_fold_rmse.csv` — per-model 5-fold RMSE used for the box plot
- `results/friedman_test.json` — paper-reported Friedman χ² = 57.30
- `results/statistical_significance_tests.csv` — Wilcoxon post-hoc vs Stacking

Outputs: `reproduced_figures/fig_statistical_significance.{pdf,png}`


In [ ]:
from IPython.display import Image, display
import pandas as pd, json
from IPython.display import Image, display
display(pd.read_csv(regen.RESULTS / 'statistical_significance_tests.csv'))
print(json.dumps(json.load(open(regen.RESULTS / 'friedman_test.json')), indent=2))
regen.fig_statistical_significance()
display(Image(filename=str(regen.FIG_DIR / 'fig_statistical_significance.png')))


---
## Ride pooling

### Cell → `tab:pooling`

Source: `results/pooling_statistics.csv`


In [ ]:
regen.table_pooling()

### Cell → `tab:pooling_baseline`

Source: `results/pooling_baselines.csv`


In [ ]:
regen.table_pooling_baseline()

### Cell → bootstrap CI figure (supplementary)

Sources: `results/bootstrap_confidence_intervals.csv`, `results/bootstrap_distributions.npz`

Not currently `\includegraphics`'d in `main.tex`, but used for the CI numbers in `tab:pooling`.


In [ ]:
from IPython.display import Image, display
regen.fig_bootstrap_confidence_intervals()
display(Image(filename=str(regen.FIG_DIR / 'fig_bootstrap_confidence_intervals.png')))

### Cell → `tab:economic`

Source: `results/economic_impact.csv`  
Scaling constants: `results/scaling_constants.json`


In [ ]:
import json
print(json.dumps(json.load(open(regen.RESULTS / 'scaling_constants.json')), indent=2))
regen.table_economic()

---
## Poolability classifier (negative result)

### Cell → `tab:confusion_compare`

Source: `results/poolability_classifier_metrics.csv`


In [ ]:
regen.table_confusion_compare()

### Cell → `fig:poolability`

Sources:
- `results/poolability_rf_summary.json` — confusion matrix, AUC, AP
- `results/poolability_roc_curve.csv` / `poolability_pr_curve.csv`
- `results/poolability_feature_importance.csv`

Outputs: `reproduced_figures/fig_poolability_classifier_full.{pdf,png}`


In [ ]:
from IPython.display import Image, display
regen.fig_poolability_classifier_full()
display(Image(filename=str(regen.FIG_DIR / 'fig_poolability_classifier_full.png')))

---
## Other frozen tables / sensitivity

### Cell → `tab:hyperparams`

Source: `results/hyperparameter_table.csv`


In [ ]:
regen.table_hyperparams()

### Cell → cluster sensitivity figure (supplementary; supports K=40 claim in Sec. spatial)

Source: `results/cluster_sensitivity.csv`


In [ ]:
from IPython.display import Image, display
regen.fig_cluster_sensitivity()
display(Image(filename=str(regen.FIG_DIR / 'fig_cluster_sensitivity.png')))

---
## Regenerate everything

Run the next cell to rebuild all tables and figures in one go (same as `python scripts/regenerate_all.py`).


In [ ]:
regen.main([])

## What is NOT regenerated here

Manual architecture diagrams (Draw.io / PowerPoint), already stored under `Taxi_Clubbing (1)/Figures/`:

| Label | File |
|-------|------|
| `fig:intro` | `Intro.pdf` |
| `fig:pipeline` | `complete_pipeline.pdf` |
| `fig:ensemble` | `ensemble_architecture.pdf` |
| `fig:hybrid` | `hybrid_architecture.pdf` |
| `fig:system_overview` | `new.pdf` |

Literature / descriptive tables (`tab:litreview`, `tab:dataset`, `tab:features`, `tab:individual_models`, `tab:sota`) are also frozen as CSV under `results/` where useful (`dataset_stats.csv`, `sota_comparison.csv`) but do not require figure regeneration.
